# Lesson 6 — Branch and Bound, Conceptually

## Learning objectives

1. Trace by hand a B&B tree for a small MILP.
2. Distinguish *bounding*, *branching*, and *node selection*.
3. Read a `discopt` solve result (`node_count`, `bound`, `gap`, `algorithm_route`) and identify when nodes are pruned by bound, by integrality, or by infeasibility.
4. Reason about why some MILPs blow up the tree and others close in 1 node.

## 1. The algorithm in pseudocode

```
LB = -inf, UB = +inf, incumbent = None
push root LP relaxation onto stack

while stack non-empty:
    pop node
    solve LP relaxation -> z_LP
    if infeasible:                  prune (infeasibility)
    elif z_LP >= UB:                prune (bound)
    elif x_LP is integer:           if z_LP < UB: UB = z_LP; incumbent = x_LP
    else:                           branch on a fractional variable, push children
```

This is {cite:p}`Land1960` and {cite:p}`Dakin1965`. Modern solvers add presolve, cuts, heuristics, and clever branching/node selection on top — but this is the core.

## 2. Branching choices

When $x_j^\star$ in the LP solution is fractional ($x_j = 1.7$, say), branch:

- **Down child:** add $x_j \le 1$.
- **Up child:** add $x_j \ge 2$.

Which fractional variable to pick is the *branching rule*. Lesson 14 covers strong branching {cite:p}`Achterberg2005`. For now: most fractional, or pseudo-cost based, are common heuristics.

## 3. Node selection

Order in which to process the open nodes:

- **Best-first** (lowest LP bound first) — minimizes nodes processed.
- **Depth-first** — minimizes memory; finds incumbents fast.
- **Best-estimate / hybrid** — balances both.

Which one does `discopt` use? It depends on the route (Lesson 1):

- A **pure MILP** is handed to HiGHS by default, which runs its own node selection, presolve and cuts {cite:p}`Huangfu2018`; `r.node_count` is then HiGHS's count (0 when presolve alone settles the model, 1 when the root closes it). `m.solve(highs_options={...})` passes HiGHS's own search and logging options (e.g. `{"output_flag": True}` prints the HiGHS log, `{"presolve": "off"}`).
- `m.solve(milp_backend="native")` instead runs `discopt`'s **own MILP tree**, so you can study the search: `strategy="best_first"` (default), `"depth_first"` or `"best_estimate"` picks the next node; `branching_rule="most_fractional"`, `"pseudocost"` (default), `"least_fractional"` or `"strong"` picks the variable {cite:p}`Achterberg2005`; `milp_cuts=False` turns off the root cuts so the tree branches on the plain LP relaxation; `batch_size=1` evaluates one node at a time; and `node_callback=f` calls `f(ctx, model)` after every batch with `ctx.node_count`, `ctx.node_bound`, `ctx.incumbent_obj` and `ctx.best_bound`.
- `discopt`'s **spatial B&B** (nonconvex NLP, MINLP) accepts the same `strategy=` values {cite:p}`Achterberg2007`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# Small knapsack we'll trace by hand
v = np.array([9, 14, 13, 11])
w = np.array([5, 7, 6, 5])
W = 13

m = do.Model("k4")
x = m.binary("x", shape=(4,))
m.subject_to(w @ x <= W)
m.maximize(v @ x)
r = m.solve()

print("optimal:", r.objective, "  x*:", r.value(x).round().astype(int))
print("nodes processed:", r.node_count, "  wall time:", round(r.wall_time, 4))
print("dual bound:", r.bound, "  gap:", r.gap)
print("route:", r.algorithm_route)

# The root LP relaxation: same model, x relaxed to [0, 1].
lp = do.Model("k4_lp")
xr = lp.continuous("x", shape=(4,), lb=0, ub=1)
lp.subject_to(w @ xr <= W)
lp.maximize(v @ xr)
rr = lp.solve()
print("root LP bound:", rr.objective, "  x_LP:", rr.value(xr).round(4))


The LP relaxation bounds the knapsack at 28 with exactly one fractional item (sorted by value/weight ratio, the LP fills greedily and splits the first item that doesn't fit). The integer optimum is 27, so a textbook B&B must branch. HiGHS nevertheless reports `node_count == 0`: its presolve settles the model before any branching. To watch a real tree, the next cell runs `discopt`'s own MILP branch-and-bound with root cuts off, one node at a time, for each node-selection strategy and branching rule. Here `node_count` counts every node *created*; a child's LP is solved only when the child is selected, so until then it carries its parent's bound.

In [ ]:
# discopt's own MILP tree on the same knapsack, without root cuts.
for strategy in ["best_first", "depth_first", "best_estimate"]:
    for rule in ["most_fractional", "pseudocost", "strong"]:
        m = do.Model("k4_native")
        x = m.binary("x", shape=(4,))
        m.subject_to(w @ x <= W)
        m.maximize(v @ x)
        r = m.solve(milp_backend="native", milp_cuts=False, batch_size=1,
                    strategy=strategy, branching_rule=rule)
        probes = r.solver_stats.get("branching/strong_probe_lps", 0)
        print(f"{strategy:>13} / {rule:<15} nodes = {r.node_count:2d}   "
              f"strong-branching probe LPs = {int(probes):2d}   z* = {r.objective}")
print("route:", r.algorithm_route[:60])


## 4. Pruning logic

A node is pruned by:

- **Infeasibility:** LP relaxation has no solution.
- **Bound:** LP value is worse than the incumbent ($z_{LP} \ge \text{UB}$ for min).
- **Integrality:** LP solution is integer; update incumbent and prune.

Stronger LP relaxations → more pruning by bound → smaller tree. Hence the obsession with cuts (Lesson 13) and tight formulations (Lesson 5).

On this instance depth-first needs fewer nodes than best-first: diving finds the incumbent 27 early, and that incumbent then prunes by bound. Best-first expands nodes in order of their *parent's* bound and only holds the weak incumbent 24 (from a heuristic) while it explores, so it branches twice more. Strong branching tests each candidate by solving both child LPs. On a 4-item knapsack every rule picks the same variable, so the tree does not change and the probe LPs are pure cost. On a larger instance the trade-off between better branching decisions and the extra LP solves is real (Lesson 14).

Exercise 1 asks you to trace the textbook tree for this instance by hand.

## 5. Why some MILPs explode

Two pathologies:

- **Symmetry** — equivalent feasible solutions reached by swapping variable indices. The tree must still explore branches that lead to identical solutions {cite:p}`Margot2010`.
- **Weak bounds** — LP relaxation far above the IP optimum, so few prunings happen.

Both are addressable: symmetry breaking via lex constraints; bounds via tighter formulation, presolve, and cuts.

## References
{cite:p}`Land1960,Dakin1965` (originals), {cite:p}`Wolsey1998` (textbook), {cite:p}`Achterberg2007` (modern engineering).